In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns

import analysis_utils as base

set_size = base.set_size
pgf_with_latex = base.pgf_with_latex
doc_width_pt = base.doc_width_pt

plt.rcParams.update(pgf_with_latex)

sns.set_palette("colorblind")


## Compute time vs. instance hardness (lightsout-5x4)

Loads a trained `lightsout-5x4` checkpoint (Transformer implicit-CoT actor, adaptive
compute budget), rolls it out on fresh instances sampled from the *training*
distribution ("trained instances" - `eval=False`, easy/in-distribution goals, as
opposed to `LightsOutEnv`'s held-out hard eval goals), and for each episode records:

- the initial grid and goal grid at reset,
- the puzzle's true shortest-path length between them (the minimum number of button
  presses needed - solved exactly over GF(2), *not* the `dist` used to generate the
  goal, which need not be minimal if `LightsOutEnv`'s toggle matrix has a nontrivial
  kernel),
- the actor's realised compute time (pondering steps), averaged per action over the
  episode, matching `stoix.systems.ramdp_vpg.evaluator`'s convention.

It then looks at whether harder instances (longer shortest path) get more compute.

**Before running:** fill in `CHECKPOINT_REL_DIR`/`CHECKPOINT_UID` (and
`CHECKPOINT_MODEL_NAME` if not the default) in the config cell below once a trained
`lightsout-5x4` checkpoint is available - none is saved locally in this repo
(`save_model` is off by default; these runs were launched on the cluster - see
`ramdp_experiments/slurm/lightsout-icot.sh`). The network hyperparameters
(`HIDDEN_DIM`, `NUM_LAYERS`, ...) must exactly match the checkpoint's training run,
or `Checkpointer.restore_params` will fail on a param-shape mismatch - the defaults
here match that slurm script's `lightsout-5x4` Transformer-CoT/reinforce/adaptive[1-5]
command.

In [ ]:
import os
import sys
from pathlib import Path

# Hydra's `hydra.searchpath` for stoix's configs (`file://stoix/configs`) is
# relative to cwd, so run everything from the repo root - matching how
# lightsout_sweep.py/ff_ppo.py are always invoked.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "ramdp_experiments" else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import hydra
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from stoix.base_types import ActorCriticParams
from stoix.envs.lightsout.lightsout_env import LightsOutEnv
from stoix.envs.lightsout.lightsout_env import default_config as lightsout_default_config
from stoix.networks.base import FeedForwardCritic
from stoix.networks.base_compute import FeedForwardActorWithComputeTime as Actor
from stoix.systems.ramdp_vpg.ff_reinforce import get_distribution_act_fn_with_compute_time
from stoix.utils import make_env as environments
from stoix.utils.checkpointing import Checkpointer

In [ ]:
# --- Checkpoint - fill in once a trained lightsout-5x4 checkpoint is available.
# These three map directly onto config.logger.checkpointing.load_args (rel_dir,
# checkpoint_uid) and model_name=config.system.system_name, i.e. exactly the knobs
# `learner_setup` itself uses to restore a checkpoint - see stoix/utils/checkpointing.py.
CHECKPOINT_REL_DIR = None  # e.g. "/path/to/run_dir/checkpoints" (absolute path is fine)
CHECKPOINT_UID = None  # e.g. "20260101120000"
CHECKPOINT_MODEL_NAME = "ramdp_ff_ppo"  # config.system.system_name default for ff_ppo.py

# Skips the checkpoint restore entirely and rolls out the freshly-initialized
# (untrained) actor instead - lets the rest of the pipeline (env, GF(2) solver,
# rollout loop, analysis/plot) be smoke-tested without a trained checkpoint. A
# random policy has no reason to spend more compute on harder instances, so
# expect ~0 correlation - this only tests that the code runs end-to-end, not
# that the trained agent behaves sensibly.
USE_RANDOM_POLICY = False

# --- Env ---
GRID_SIZE = "5x4"
EPISODE_LENGTH = 10
EVAL_EPISODE_LENGTH = 20
DIFFICULTY_THRESHOLD = 0.5
# "Trained instances" = the training distribution (eval=False, easy/in-distribution
# goals - see LightsOutEnv's module docstring). Flip to True for the held-out hard
# eval instances instead.
EVAL_HARD_INSTANCES = False

# --- Network: Transformer implicit-CoT, adaptive compute budget. Must exactly match
# the checkpoint's training run (see the intro markdown cell). Defaults below are
# lightsout-5x4's Transformer-CoT/reinforce/adaptive[1-5] run (see
# ramdp_experiments/slurm/lightsout-icot.sh's active command).
HIDDEN_DIM = 128
NUM_LAYERS = 2
NUM_HEADS = 8
MLP_DIM = 256
QKV_DIM = 256
MIN_STEPS = 1
MAX_STEPS = 5
HALTING_TEMPERATURE = 5.0
USE_INPUT_LAYER_NORM = True
USE_SANDWICH_NORM = False
USE_RMSNORM = True
STOP_GRADIENT_HALTING_INPUT = False
QAC_VARIANT = "reinforce"  # plain V-only critic - must match qac_variant used to train the checkpoint

# --- Rollout ---
NUM_EVAL_EPISODES = 500
SEED = 0
EVALUATION_GREEDY = False  # match training-time stochastic policy; True evaluates the mode action instead

In [ ]:
def build_stoix_config():
    """Composes the same DictConfig ff_ppo.py's @hydra.main would build for a
    lightsout-5x4 / transformer_compute run, via the CLI overrides
    ramdp_experiments/lightsout_sweep.py's Job.command() would emit for this
    (system=ff_ppo_reinforce, architecture=transformer) combo."""
    overrides = [
        # Base env config is a fixed 3x3 example - grid size/task name/episode
        # length are overridden below, same as Job.command().
        "env=lightsout/lightsout_3x3",
        f"env.scenario.name=lightsout-{GRID_SIZE}",
        f"env.scenario.task_name=lightsout_{GRID_SIZE}",
        f"env.kwargs.episode_length={EPISODE_LENGTH}",
        f"++env.kwargs.eval_episode_length={EVAL_EPISODE_LENGTH}",
        f"env.kwargs.difficulty_threshold={DIFFICULTY_THRESHOLD}",
        # LightsOutEnv's native (m, n, 2) observation is flattened for non-CNN
        # torsos (transformer_compute has no CNN input_layer) - see
        # lightsout_sweep.py's Job.command().
        "+env.wrapper._target_=stoa.FlattenObservationWrapper",
        "network=transformer_compute",
        f"network.actor_network.pre_torso.hidden_dim={HIDDEN_DIM}",
        f"++network.actor_network.pre_torso.num_layers={NUM_LAYERS}",
        f"++network.actor_network.pre_torso.num_heads={NUM_HEADS}",
        f"++network.actor_network.pre_torso.mlp_dim={MLP_DIM}",
        f"++network.actor_network.pre_torso.qkv_dim={QKV_DIM}",
        f"network.actor_network.pre_torso.min_steps={MIN_STEPS}",
        f"network.actor_network.pre_torso.max_steps={MAX_STEPS}",
        f"network.actor_network.pre_torso.halting_temperature={HALTING_TEMPERATURE}",
        f"++network.actor_network.pre_torso.use_input_layer_norm={USE_INPUT_LAYER_NORM}",
        f"++network.actor_network.pre_torso.use_sandwich_norm={USE_SANDWICH_NORM}",
        f"++network.actor_network.pre_torso.use_rmsnorm={USE_RMSNORM}",
        f"++network.actor_network.pre_torso.stop_gradient_halting_input={STOP_GRADIENT_HALTING_INPUT}",
        f"system.qac_variant={QAC_VARIANT}",
        f"arch.evaluation_greedy={EVALUATION_GREEDY}",
    ]
    GlobalHydra.instance().clear()
    with initialize_config_dir(
        config_dir=str(REPO_ROOT / "stoix/configs/default/anakin"), version_base=None
    ):
        config = compose(config_name="default_ramdp_ff_ppo.yaml", overrides=overrides)
    return config


config = build_stoix_config()

In [ ]:
def build_actor_and_restore(config, seed=0):
    """Mirrors the actor/critic construction in
    stoix.systems.ramdp_vpg.ff_ppo.learner_setup (skipping the
    optimizer/learner_fn machinery, which pure inference doesn't need), then
    either restores the actor's trained params from CHECKPOINT_REL_DIR/UID or,
    if USE_RANDOM_POLICY, returns the freshly-initialized params as-is. Returns
    (env, eval_env, act_fn, actor_params)."""
    assert USE_RANDOM_POLICY or CHECKPOINT_UID is not None, (
        "Fill in CHECKPOINT_REL_DIR/CHECKPOINT_UID in the config cell before running this "
        "(or set USE_RANDOM_POLICY=True to smoke-test the pipeline without one)."
    )

    env, eval_env = environments.make(config=config)
    num_actions = int(env.action_space().num_values)

    key = jax.random.PRNGKey(seed)
    actor_net_key, critic_net_key = jax.random.split(key)

    actor_torso = hydra.utils.instantiate(config.network.actor_network.pre_torso)
    actor_action_head = hydra.utils.instantiate(
        config.network.actor_network.action_head, action_dim=num_actions
    )
    actor_network = Actor(torso=actor_torso, action_head=actor_action_head)

    init_x = eval_env.observation_space().generate_value()
    init_x = jax.tree_util.tree_map(lambda x: x[None, ...], init_x)
    actor_params = actor_network.init(actor_net_key, init_x, torso_kwargs={"rng": actor_net_key})

    if not USE_RANDOM_POLICY:
        # QAC_VARIANT="reinforce" -> plain V-only critic (FeedForwardCritic), matching
        # ff_ppo.py's learner_setup. The critic is only needed to reconstruct the
        # checkpoint's full ActorCriticParams structure for Checkpointer.restore_params -
        # its params are discarded below, since rollouts here only use the actor.
        critic_torso = hydra.utils.instantiate(config.network.critic_network.pre_torso)
        critic_head = hydra.utils.instantiate(config.network.critic_network.critic_head)
        critic_network = FeedForwardCritic(torso=critic_torso, critic_head=critic_head)
        critic_params = critic_network.init(critic_net_key, init_x)
        params = ActorCriticParams(actor_params, critic_params)

        loaded_checkpoint = Checkpointer(
            model_name=CHECKPOINT_MODEL_NAME,
            rel_dir=CHECKPOINT_REL_DIR or "checkpoints",
            checkpoint_uid=CHECKPOINT_UID,
        )
        restored_params, _ = loaded_checkpoint.restore_params(input_params=params)
        actor_params = restored_params.actor_params

    act_fn = get_distribution_act_fn_with_compute_time(config, actor_network.apply)
    return env, eval_env, act_fn, actor_params


env, eval_env, act_fn, actor_params = build_actor_and_restore(config, seed=SEED)
rollout_env = eval_env if EVAL_HARD_INSTANCES else env

### Shortest path between grids (GF(2) minimum-weight solve)

Pressing a button toggles a fixed set of cells; pressing it twice cancels out, and
press order doesn't matter, so reaching `goal` from `initial` is exactly solving
`action_matrix @ x = (initial XOR goal) (mod 2)` for a 0/1 button vector `x`, and the
puzzle's shortest-path length is `x`'s minimum Hamming weight over all solutions
(there can be more than one if `action_matrix` has a nontrivial kernel - see
`LightsOutEnv`'s module docstring on why the goal-generating `dist` isn't necessarily
minimal). `gf2_prepare` row-reduces `action_matrix` once; `gf2_min_weight_solution`
then does cheap per-instance back-substitution plus a brute-force search over the
(typically tiny) kernel to find the minimum-weight solution.

In [ ]:
def gf2_prepare(A, max_kernel_dim=20):
    """Row-reduces the (n, n) GF(2) matrix `A`, tracking the elementary row
    operations (via an augmented identity block) so they can be reapplied to
    any right-hand side later. Returns (RREF, T, pivot_cols, free_cols, rank)
    where T @ A == RREF (mod 2). Raises if the kernel is too large to brute-
    force (see gf2_min_weight_solution) - not expected for Lights Out grids
    of the sizes used here."""
    n = A.shape[0]
    M = np.concatenate([A.astype(np.uint8) % 2, np.eye(n, dtype=np.uint8)], axis=1)
    pivot_cols = []
    row = 0
    for col in range(n):
        pivot = next((r for r in range(row, n) if M[r, col]), None)
        if pivot is None:
            continue
        M[[row, pivot]] = M[[pivot, row]]
        for r in range(n):
            if r != row and M[r, col]:
                M[r] ^= M[row]
        pivot_cols.append(col)
        row += 1
        if row == n:
            break
    rank = row
    free_cols = [c for c in range(n) if c not in pivot_cols]
    assert len(free_cols) <= max_kernel_dim, (
        f"action_matrix's kernel has {len(free_cols)} dimensions - too many to "
        "brute-force minimum-weight solutions over."
    )
    return M[:, :n], M[:, n:], pivot_cols, free_cols, rank


def gf2_min_weight_solution(prepared, diff):
    """Minimum-Hamming-weight x s.t. action_matrix @ x == diff (mod 2), or
    None if infeasible. `prepared` is gf2_prepare(action_matrix)'s output."""
    RREF, T, pivot_cols, free_cols, rank = prepared
    n = RREF.shape[0]
    rd = (T @ diff.astype(np.uint8)) % 2
    if np.any((RREF[rank:].sum(axis=1) == 0) & (rd[rank:] != 0)):
        return None

    x0 = np.zeros(n, dtype=np.uint8)
    for i, col in enumerate(pivot_cols):
        x0[col] = rd[i]

    kernel_basis = []
    for fc in free_cols:
        v = np.zeros(n, dtype=np.uint8)
        v[fc] = 1
        for i, col in enumerate(pivot_cols):
            v[col] = RREF[i, fc]
        kernel_basis.append(v)

    best, best_weight = x0, int(x0.sum())
    for mask in range(1, 1 << len(kernel_basis)):
        v = x0.copy()
        for i, basis_vec in enumerate(kernel_basis):
            if mask & (1 << i):
                v ^= basis_vec
        weight = int(v.sum())
        if weight < best_weight:
            best, best_weight = v, weight
    return best_weight


def make_action_matrix(grid_size):
    m, n = (int(x) for x in grid_size.split("x"))
    cfg = lightsout_default_config()
    cfg.m, cfg.n = m, n
    return np.array(LightsOutEnv(cfg, eval=False).action_matrix)


action_matrix = make_action_matrix(GRID_SIZE)
gf2_prepared = gf2_prepare(action_matrix)
print(f"action_matrix kernel dimension: {len(gf2_prepared[3])}")

In [ ]:
def rollout_episodes(env, act_fn, params, num_episodes, seed):
    """Runs `num_episodes` non-vectorized episodes against `env` (a single,
    un-batched LightsOutEnv instance - stoix.utils.make_env's `eval_env` is
    never wrapped with the train env's VmapWrapper/AutoResetWrapper, see
    make_lightsout_env), recording each episode's initial/goal grids and the
    actor's mean compute time per action (matching
    stoix.systems.ramdp_vpg.evaluator's `episode_compute_time / step_count`
    convention)."""
    key = jax.random.PRNGKey(seed)
    records = []
    for episode in range(num_episodes):
        key, reset_key = jax.random.split(key)
        state, timestep = env.reset(reset_key)
        initial_grid = np.array(state.data.grid)
        goal_grid = np.array(state.data.goal)

        total_compute_time = 0.0
        step_count = 0
        while not timestep.last():
            key, act_key = jax.random.split(key)
            obs = jax.tree_util.tree_map(lambda x: x[jnp.newaxis, ...], timestep.observation)
            action, compute_time, _, _ = act_fn(params, obs, act_key)
            state, timestep = env.step(state, action.squeeze(0))
            total_compute_time += float(np.array(compute_time).squeeze())
            step_count += 1

        records.append(
            {
                "episode": episode,
                "initial_grid": initial_grid,
                "goal_grid": goal_grid,
                "compute_time": total_compute_time / step_count,
                "episode_length": step_count,
                "solved": bool(np.array(timestep.reward).squeeze() >= 1.0),
            }
        )
    return records


records = rollout_episodes(rollout_env, act_fn, actor_params, NUM_EVAL_EPISODES, seed=SEED)
for r in records:
    diff = np.logical_xor(r["initial_grid"], r["goal_grid"]).astype(np.uint8)
    r["shortest_path_length"] = gf2_min_weight_solution(gf2_prepared, diff)

df = pd.DataFrame.from_records(records)
n_infeasible = df["shortest_path_length"].isna().sum()
if n_infeasible:
    print(f"{n_infeasible}/{len(df)} episodes had no feasible button-press solution (unexpected).")
df.head()

In [ ]:
CI95_Z = 1.96  # normal-approximation 95% CI half-width, in units of SEM

plot_df = df.dropna(subset=["shortest_path_length"])

pearson_r = plot_df["shortest_path_length"].corr(plot_df["compute_time"], method="pearson")
spearman_r = plot_df["shortest_path_length"].corr(plot_df["compute_time"], method="spearman")
print(f"Pearson r  = {pearson_r:.3f}")
print(f"Spearman r = {spearman_r:.3f}")

by_distance = plot_df.groupby("shortest_path_length")["compute_time"].agg(["mean", "sem", "count"])

fig, ax = plt.subplots(figsize=set_size(doc_width_pt, fraction=0.6))
rng = np.random.default_rng(0)
jitter = rng.uniform(-0.15, 0.15, size=len(plot_df))
ax.scatter(
    plot_df["shortest_path_length"] + jitter,
    plot_df["compute_time"],
    s=10,
    alpha=0.25,
    color=sns.color_palette("colorblind")[0],
    label="episode",
)
ax.errorbar(
    by_distance.index,
    by_distance["mean"],
    yerr=CI95_Z * by_distance["sem"].fillna(0.0),
    marker="o",
    ms=4,
    linewidth=1.5,
    color=sns.color_palette("colorblind")[3],
    label="mean +/- 95% CI",
)
ax.set_xlabel("Shortest path length (button presses)")
ax.set_ylabel("Mean compute time per action")
ax.set_title(f"lightsout-{GRID_SIZE}: hardness vs. compute time (Pearson r={pearson_r:.2f})")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.savefig("analysis-compute_time-hardness.pdf", dpi=600, format="pdf", bbox_inches="tight")